In [2]:
# 嵌入模型下载
from modelscope.hub.snapshot_download import snapshot_download

embed_dir = snapshot_download(
    "Qwen/Qwen3-Embedding-0.6B",
    cache_dir=r"D:\WorkSystem\Large-model-learning\LangChain-tutorial\Model\Embedding_Model",  # 你想放哪都行
)
print(embed_dir)

d:\CodeRelated\codeService\anaconda3\envs\LangChainPy310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

# 1.数据的存储
举例1：从TXT文档中加载数据，向量化后存储到Chroma数据库中

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings


loader = TextLoader(
    file_path="./asset/load/09-ai1.txt",
    encoding="utf-8",
)

docs = loader.load()


# 创建文本拆分其
text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
)

splitter_docs = text_splitter.split_documents(docs)

print(len(splitter_docs))

# 创建嵌入模型


embedding_model = HuggingFaceEmbeddings(
    model_name="D:\WorkSystem\Large-model-learning\LangChain-tutorial\Model\Embedding_Model\Qwen\Qwen3-Embedding-0___6B",  # 👈 这里填 snapshot_download 返回的目录
    model_kwargs={"device": "cpu"},  # 有 GPU 可改成 "cuda"
    encode_kwargs={"normalize_embeddings": True},  # 检索更常用
)

# 将文档及嵌入模型传入到Chroma相关的结构中，进行数据的存储
db = Chroma.from_documents(
    documents=splitter_docs,
    embedding=embedding_model,
)

C:\Users\javis\AppData\Local\Temp\ipykernel_44456\2958553807.py:28: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


3


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 817.89it/s, Materializing param=norm.weight]                              


### 数据到底存储在哪里了？
如果我们使用的from_documents()中没有显示的指明存储位置的话，则将当前的数据存储在内存中，并缓存起来，如果需要知名具体的存储位置，需要设置参数persisit_directory的值

In [ ]:
db = Chroma.from_documents(
  documents=splitter_docs,
  embedding=embedding_model,
  persist_directory="./asset/chroma-1"
)

### 需求明确，在向量数据库中，不仅存储了数据的向量，还存储了数据本身

In [8]:
query = "人工智能的核心技术有哪些呢？"

docs = db.similarity_search(query)
# print(docs[0].page_content)

## 操作CSV文件并向量化

In [ ]:
from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    CSVLoader,
)

loader = CSVLoader(
    "./asset/load/03-load.csv",
)
pages = loader.load_and_split()


# 文本拆分
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(chunk_size=500)
docs = text_splitter.split_documents(pages)

db_path = "./chroma_db"
db = Chroma.from_documents(
    docs,
    embedding_model,
    persist_directory=db_path,
)

In [20]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

raw_document = [
    Document(
        page_content="葡萄是一种常见的水果，属于葡萄科葡萄属植物。它的果实呈圆形或椭圆形，颜色有绿色、紫色、红色等多种。葡萄富含维生素C和抗氧化物质，可以直接食用或酿造成葡萄酒。",
        metadata={"source": "水果", "type": "植物"},
    ),
    Document(
        page_content="白菜是十字花科蔬菜，原产于中国北方。它的叶片层层包裹形成紧密的球状，口感清脆微甜。白菜富含膳食纤维和维生素K，常用于制作泡菜、炒菜或煮汤。",
        metadata={"source": "蔬菜", "type": "植物"},
    ),
    Document(
        page_content="狗是人类最早驯化的动物之一，属于犬科。它们具有高度社会性，能理解人类情绪，常被用作宠物、导盲犬或警犬。不同品种的狗在体型、毛色和性格上有很大差异。",
        metadata={"source": "动物", "type": "哺乳动物"},
    ),
    Document(
        page_content="猫是小型肉食性哺乳动物，性格独立但也能与人类建立亲密关系。它们夜视能力极强，擅长捕猎老鼠。家猫的品种包括波斯猫、暹罗猫等，毛色和花纹多样。",
        metadata={"source": "动物", "type": "哺乳动物"},
    ),
    Document(
        page_content="人类是地球上最具智慧的生物，属于灵长目人科。现代人类（智人）拥有高度发达的大脑，创造了语言、工具和文明。人类的平均寿命约70-80年，分布在全球各地。",
        metadata={"source": "生物", "type": "灵长类"},
    ),
    Document(
        page_content="太阳是太阳系的中心恒星，直径约139万公里，主要由氢和氦组成。它通过核聚变反应产生能量，为地球提供光和热。太阳活动周期约为11年，会影响地球气候。",
        metadata={"source": "天文", "type": "恒星"},
    ),
    Document(
        page_content="长城是中国古代的军事防御工程，总长度超过2万公里。它始建于春秋战国时期，秦朝连接各段，明朝大规模重修。长城是世界文化遗产和人类建筑奇迹。",
        metadata={"source": "历史", "type": "建筑"},
    ),
    Document(
        page_content="量子力学是研究微观粒子运动规律的物理学分支。它提出了波粒二象性、测不准原理等概念，彻底改变了人类对物质世界的认知。量子计算机正是基于这一理论发展而来。",
        metadata={"source": "物理", "type": "科学"},
    ),
    Document(
        page_content="《红楼梦》是中国古典文学四大名著之一，作者曹雪芹。小说以贾、史、王、薛四大家族的兴衰为背景，描绘了贾宝玉与林黛玉的爱情悲剧，反映了封建社会的种种矛盾。",
        metadata={"source": "文学", "type": "小说"},
    ),
    Document(
        page_content="新冠病毒（SARS-CoV-2）是一种可引起呼吸道疾病的冠状病毒。它通过飞沫传播，主要症状包括发热、咳嗽、乏力。疫苗和戴口罩是有效的预防措施。",
        metadata={"source": "医学", "type": "病毒"},
    ),
]
embedding_model = HuggingFaceEmbeddings(
    model_name="D:\WorkSystem\Large-model-learning\LangChain-tutorial\Model\Embedding_Model\Qwen\Qwen3-Embedding-0___6B",  # 👈 这里填 snapshot_download 返回的目录
    model_kwargs={"device": "cpu"},  # 有 GPU 可改成 "cuda"
    encode_kwargs={"normalize_embeddings": True},  # 检索更常用
)
db = Chroma.from_documents(
    documents=raw_document,
    embedding=embedding_model,
    persist_directory="./asset/chroma-3",
)

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 736.39it/s, Materializing param=norm.weight]                              


In [21]:
# 检索示例（返回3个最相关的结果）
query = "哺乳动物"
docs = db.similarity_search(
    query,
    k=3,  # k=3表示返回3个相关文档
)
print(f"查询：'{query}'的结果：")
for i, doc in enumerate(docs, 1):
    print(f"\n结果{i}:")
    print(f"内容：{doc.page_content}")
    print(f"原数据：{doc.metadata}")

查询：'哺乳动物'的结果：

结果1:
内容：狗是人类最早驯化的动物之一，属于犬科。它们具有高度社会性，能理解人类情绪，常被用作宠物、导盲犬或警犬。不同品种的狗在体型、毛色和性格上有很大差异。
原数据：{'type': '哺乳动物', 'source': '动物'}

结果2:
内容：狗是人类最早驯化的动物之一，属于犬科。它们具有高度社会性，能理解人类情绪，常被用作宠物、导盲犬或警犬。不同品种的狗在体型、毛色和性格上有很大差异。
原数据：{'type': '哺乳动物', 'source': '动物'}

结果3:
内容：猫是小型肉食性哺乳动物，性格独立但也能与人类建立亲密关系。它们夜视能力极强，擅长捕猎老鼠。家猫的品种包括波斯猫、暹罗猫等，毛色和花纹多样。
原数据：{'type': '哺乳动物', 'source': '动物'}


In [15]:
# 检索示例（返回3个最相关的结果）
query = "哺乳动物"
embedding_vector = embedding_model.embed_query(query)
docs = db.similarity_search_by_vector(
    embedding_vector,
    k=3,  # k=3表示返回3个相关文档
)
print(f"查询：'{query}'的结果：")
for i, doc in enumerate(docs, 1):
    print(f"\n结果{i}:")
    print(f"内容：{doc.page_content}")
    print(f"原数据：{doc.metadata}")

查询：'哺乳动物'的结果：

结果1:
内容：3. 人工智能的核心技术
3.1 机器学习
机器学习是人工智能的核心技术之一，通过算法使计算机从数据中学习并做出决策。常见的机器学习算法包括监督学习、无监督学习和强化学习。监督学习通过标记数据进行训练，无监督学习则从未标记数据中寻找模式，强化学习则通过与环境交互来优化决策。
3.2 深度学习
深度学习是机器学习的一个子领域，通过多层神经网络进行特征提取和模式识别。深度学习在图像识别、自然语言处理、语音识别等领域取得了显著成果。常见的深度学习模型包括卷积神经网络（CNN）、循环神经网络（RNN）和长短期记忆网络（LSTM）。
3.3 自然语言处理
自然语言处理（NLP）是人工智能的一个重要分支，致力于使计算机能够理解和生成人类语言。NLP技术广泛应用于机器翻译、情感分析、文本分类等领域。近年来，基于深度学习的NLP模型（如BERT、GPT）在语言理解任务中取得了突破性进展。
3.4 计算机视觉
计算机视觉是人工智能的另一个重要分支，致力于使计算机能够理解和处理图像和视频。计算机视觉技术广泛应用于图像识别、目标检测、人脸识别等领域。深度学习模型（如CNN）在计算机视觉任务中取得了显著成果。

4. 人工智能的应用领域
4.1 医疗健康
人工智能在医疗健康领域的应用包括疾病诊断、药物研发、个性化医疗等。通过分析医学影像和患者数据，人工智能可以帮助医生更准确地诊断疾病，提高治疗效果。
4.2 金融
人工智能在金融领域的应用包括风险评估、欺诈检测、算法交易等。通过分析市场数据和交易记录，人工智能可以帮助金融机构做出更明智的决策，提高运营效率。
4.3 教育
人工智能在教育领域的应用包括个性化学习、智能辅导、自动评分等。通过分析学生的学习数据，人工智能可以为学生提供个性化的学习建议，提高学习效果。
4.4 交通
人工智能在交通领域的应用包括自动驾驶、交通管理、智能导航等。通过分析交通数据和路况信息，人工智能可以帮助优化交通流量，提高交通安全。
原数据：{'source': './asset/load/09-ai1.txt'}

结果2:
内容：3. 人工智能的核心技术
3.1 机器学习
机器学习是人工智能的核心技术之一，通过算法使计算机从数据中学习并做出决策。常见的机器学习算法包括监督学习、无监督学习和强化学习。监督学习通过标记数据

In [22]:
# 检索示例（返回3个最相关的结果）
query = "哺乳动物"
embedding_vector = embedding_model.embed_query(query)
docs = db.similarity_search_by_vector(
    embedding_vector,
    k=3,  # k=3表示返回3个相关文档
    filter={"source":"动物"}
)
print(f"查询：'{query}'的结果：")
for i, doc in enumerate(docs, 1):
    print(f"\n结果{i}:")
    print(f"内容：{doc.page_content}")
    print(f"原数据：{doc.metadata}")

查询：'哺乳动物'的结果：

结果1:
内容：狗是人类最早驯化的动物之一，属于犬科。它们具有高度社会性，能理解人类情绪，常被用作宠物、导盲犬或警犬。不同品种的狗在体型、毛色和性格上有很大差异。
原数据：{'type': '哺乳动物', 'source': '动物'}

结果2:
内容：狗是人类最早驯化的动物之一，属于犬科。它们具有高度社会性，能理解人类情绪，常被用作宠物、导盲犬或警犬。不同品种的狗在体型、毛色和性格上有很大差异。
原数据：{'source': '动物', 'type': '哺乳动物'}

结果3:
内容：猫是小型肉食性哺乳动物，性格独立但也能与人类建立亲密关系。它们夜视能力极强，擅长捕猎老鼠。家猫的品种包括波斯猫、暹罗猫等，毛色和花纹多样。
原数据：{'type': '哺乳动物', 'source': '动物'}


In [23]:
# 检索示例（返回3个最相关的结果）
query = "哺乳动物"
docs = db.similarity_search(
    query = query,
    k=3,  # k=3表示返回3个相关文档
    filter=({"source": "动物"}),
)
print(f"查询：'{query}'的结果：")
for i, doc in enumerate(docs, 1):
    print(f"\n结果{i}:")
    print(f"内容：{doc.page_content}")
    print(f"原数据：{doc.metadata}")

查询：'哺乳动物'的结果：

结果1:
内容：狗是人类最早驯化的动物之一，属于犬科。它们具有高度社会性，能理解人类情绪，常被用作宠物、导盲犬或警犬。不同品种的狗在体型、毛色和性格上有很大差异。
原数据：{'type': '哺乳动物', 'source': '动物'}

结果2:
内容：狗是人类最早驯化的动物之一，属于犬科。它们具有高度社会性，能理解人类情绪，常被用作宠物、导盲犬或警犬。不同品种的狗在体型、毛色和性格上有很大差异。
原数据：{'source': '动物', 'type': '哺乳动物'}

结果3:
内容：猫是小型肉食性哺乳动物，性格独立但也能与人类建立亲密关系。它们夜视能力极强，擅长捕猎老鼠。家猫的品种包括波斯猫、暹罗猫等，毛色和花纹多样。
原数据：{'source': '动物', 'type': '哺乳动物'}
